# 01. Duomenų paruošimas (UCI HAR)

Šiame etape **nekuriame klasifikavimo modelių**. Tikslas — tvarkingai užkrauti oficialų UCI HAR rinkinį, sujungti originalias train/test dalis ir sudaryti **naują TRAIN/TEST skaidymą pagal žmones (subject ID)**.

Vėliau modeliai bus vertinami pagal tai, ar jie atpažįsta **mokymo metu nematyto žmogaus** veiklą.

## Kas daroma ir kodėl

Užkrauname visus 561 požymius, veiklos klases ir kiekvieno įrašo `subject` ID iš lokalaus UCI aplanko.

Tai darome todėl, kad visos tolesnės patikros ir skaidymas turi remtis **realiais failais**, o ne išgalvotais skaičiais. `RANDOM_STATE = 42` fiksuoja atsitiktinumą, kad skaidymas būtų atkuriamas.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"RANDOM_STATE = {RANDOM_STATE}")

RANDOM_STATE = 42


## Lokalaus UCI HAR rinkinio kelias

Naudojame tik projekte jau esančius oficialius UCI failus. Archyvas šiame kompiuteryje išpakuotas giliau nei `data/UCI HAR Dataset/`, todėl kelias randamas pagal `features.txt`, o ne įrašomas kaip spėjimas.

In [2]:
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
project_root = None
for cand in candidates:
    if (cand / "data").exists() and (cand / "notebooks").exists():
        project_root = cand
        break
if project_root is None:
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

preferred = [
    project_root / "data" / "UCI HAR Dataset",
    project_root
    / "data"
    / "human+activity+recognition+using+smartphones (2)"
    / "UCI HAR Dataset"
    / "UCI HAR Dataset",
]

data_dir = None
for path in preferred:
    if (path / "features.txt").exists() and (path / "train" / "X_train.txt").exists():
        data_dir = path
        break

if data_dir is None:
    matches = [
        p
        for p in (project_root / "data").rglob("features.txt")
        if "__MACOSX" not in str(p)
    ]
    if not matches:
        raise FileNotFoundError(
            "Nerastas lokalus UCI HAR rinkinys po data/. "
            "Turi būti features.txt, train/ ir test/."
        )
    data_dir = matches[0].parent

print(f"Projekto šaknis: {project_root}")
print(f"Duomenų aplankas: {data_dir}")
print(f"Kelias egzistuoja: {data_dir.exists()}")

Projekto šaknis: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26
Duomenų aplankas: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\data\human+activity+recognition+using+smartphones (2)\UCI HAR Dataset\UCI HAR Dataset
Kelias egzistuoja: True


## Failų užkrovimas

Skaitome `features.txt`, `activity_labels.txt` ir originalias UCI `train/` bei `test/` lenteles. UCI HAR `features.txt` turi pasikartojančių požymių vardų, todėl vardus padarome unikalius — tai tik stulpelių pavadinimai, požymių reikšmių nekeičiame.

In [3]:
def make_unique_names(names):
    seen = {}
    unique = []
    for name in names:
        count = seen.get(name, 0) + 1
        seen[name] = count
        unique.append(name if count == 1 else f"{name}_{count}")
    return unique


def load_feature_table(path, feature_names):
    return pd.read_csv(
        path,
        sep=r"\s+",
        header=None,
        names=feature_names,
        engine="python",
    )


def load_vector(path, column_name):
    return pd.read_csv(path, sep=r"\s+", header=None, names=[column_name], engine="python")


features_meta = pd.read_csv(
    data_dir / "features.txt",
    sep=r"\s+",
    header=None,
    names=["feature_id", "feature_name"],
    engine="python",
)
activity_labels = pd.read_csv(
    data_dir / "activity_labels.txt",
    sep=r"\s+",
    header=None,
    names=["activity_id", "activity_name"],
    engine="python",
)

feature_names = make_unique_names(features_meta["feature_name"].tolist())

X_train_orig = load_feature_table(data_dir / "train" / "X_train.txt", feature_names)
y_train_orig = load_vector(data_dir / "train" / "y_train.txt", "activity_id")
subject_train_orig = load_vector(data_dir / "train" / "subject_train.txt", "subject_id")

X_test_orig = load_feature_table(data_dir / "test" / "X_test.txt", feature_names)
y_test_orig = load_vector(data_dir / "test" / "y_test.txt", "activity_id")
subject_test_orig = load_vector(data_dir / "test" / "subject_test.txt", "subject_id")

print("Požymių meta eilučių:", len(features_meta))
print("Unikalių požymių vardų po sutvarkymo:", len(feature_names))
print("Veiklos etiketės:")
print(activity_labels.to_string(index=False))
print()
print("Originalus UCI TRAIN:", X_train_orig.shape, y_train_orig.shape, subject_train_orig.shape)
print("Originalus UCI TEST :", X_test_orig.shape, y_test_orig.shape, subject_test_orig.shape)

Požymių meta eilučių: 561
Unikalių požymių vardų po sutvarkymo: 561
Veiklos etiketės:
 activity_id      activity_name
           1            WALKING
           2   WALKING_UPSTAIRS
           3 WALKING_DOWNSTAIRS
           4            SITTING
           5           STANDING
           6             LAYING

Originalus UCI TRAIN: (7352, 561) (7352, 1) (7352, 1)
Originalus UCI TEST : (2947, 561) (2947, 1) (2947, 1)


## Originalių UCI dalių sujungimas

Sujungiame oficialias train ir test dalis į vieną rinkinį.

Tai darome todėl, kad šiame eksperimente **nenaudosime** UCI pasiūlyto žmonių paskirstymo. Vietoj to visus 30 žmonių turime vienoje vietoje ir patys sudarysime fiksuotą skaidymą pagal `subject ID`.

In [4]:
X = pd.concat([X_train_orig, X_test_orig], axis=0, ignore_index=True)
y = pd.concat([y_train_orig, y_test_orig], axis=0, ignore_index=True)
subjects = pd.concat([subject_train_orig, subject_test_orig], axis=0, ignore_index=True)

y_named = y.merge(activity_labels, on="activity_id", how="left")

n_records = len(X)
n_features = X.shape[1]
n_subjects = subjects["subject_id"].nunique()
n_missing = int(X.isna().sum().sum() + y.isna().sum().sum() + subjects.isna().sum().sum())
activity_ids = sorted(y["activity_id"].unique().tolist())
activity_names = activity_labels.set_index("activity_id").loc[activity_ids, "activity_name"].tolist()

print(f"Bendras įrašų skaičius: {n_records}")
print(f"Požymių skaičius: {n_features}")
print(f"X dimensija: {X.shape}")
print(f"y dimensija: {y.shape}")
print(f"subject dimensija: {subjects.shape}")
print(f"Veiklos klasių ID: {activity_ids}")
print(f"Visos veiklos klasės: {activity_names}")
print(f"Unikalių subject ID skaičius: {n_subjects}")
print(f"Subject ID: {sorted(subjects['subject_id'].unique().tolist())}")
print(f"Trūkstamų reikšmių viso: {n_missing}")
print(f"Ar nėra trūkstamų reikšmių: {n_missing == 0}")

Bendras įrašų skaičius: 10299
Požymių skaičius: 561
X dimensija: (10299, 561)
y dimensija: (10299, 1)
subject dimensija: (10299, 1)
Veiklos klasių ID: [1, 2, 3, 4, 5, 6]
Visos veiklos klasės: ['WALKING', 'WALKING_UPSTAIRS', 'WALKING_DOWNSTAIRS', 'SITTING', 'STANDING', 'LAYING']
Unikalių subject ID skaičius: 30
Subject ID: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
Trūkstamų reikšmių viso: 0
Ar nėra trūkstamų reikšmių: True


## Naujas TRAIN/TEST skaidymas pagal žmones

Bendrą rinkinį dalijame maždaug **70 % / 30 % pagal subject ID**, ne pagal atsitiktines eilutes.

`GroupShuffleSplit` visus vieno žmogaus įrašus palieka toje pačioje dalyje. Tai svarbu HAR eksperimente: to paties žmogaus eisena, laikysena ir telefono nešiojimo būdas yra panašūs. Jei tas pats `subject` patektų ir į TRAIN, ir į TEST, modelis galėtų „atsiminti“ žmogų, o ne išmokti veiklos požymius. Tikslas — patikrinti gebėjimą atpažinti **naujo, mokymo metu nematyto žmogaus** veiklą.

Pastaba: 70/30 taikoma **žmonėms**, todėl įrašų skaičius gali šiek tiek skirtis nuo 70/30, nes skirtingi žmonės turi skirtingą langų skaičių.

In [5]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y["activity_id"], groups=subjects["subject_id"]))

X_train_final = X.iloc[train_idx].reset_index(drop=True)
X_test_final = X.iloc[test_idx].reset_index(drop=True)
y_train_final = y.iloc[train_idx].reset_index(drop=True)
y_test_final = y.iloc[test_idx].reset_index(drop=True)
subjects_train_final = subjects.iloc[train_idx].reset_index(drop=True)
subjects_test_final = subjects.iloc[test_idx].reset_index(drop=True)

print("X_train_final:", X_train_final.shape)
print("X_test_final :", X_test_final.shape)
print("y_train_final:", y_train_final.shape)
print("y_test_final :", y_test_final.shape)
print("subjects_train_final:", subjects_train_final.shape)
print("subjects_test_final :", subjects_test_final.shape)
print()
print(
    "Įrašų dalis TRAIN:",
    round(len(X_train_final) / len(X), 4),
    "| TEST:",
    round(len(X_test_final) / len(X), 4),
)

X_train_final: (7144, 561)
X_test_final : (3155, 561)
y_train_final: (7144, 1)
y_test_final : (3155, 1)
subjects_train_final: (7144, 1)
subjects_test_final : (3155, 1)

Įrašų dalis TRAIN: 0.6937 | TEST: 0.3063


## Data leakage patikrinimas

Programiškai tikriname, kad TRAIN ir TEST žmonių aibės **nesikirstų**. Jei bent vienas `subject ID` būtų abiejose dalyse, tai būtų duomenų nutekėjimas (leakage): testas nebebūtų „naujas žmogus“.

Taip pat žiūrime klasių pasiskirstymą — visos 6 veiklos turi būti ir TRAIN, ir TEST.

In [6]:
train_subjects = sorted(subjects_train_final["subject_id"].unique().tolist())
test_subjects = sorted(subjects_test_final["subject_id"].unique().tolist())
subject_overlap = set(train_subjects).intersection(set(test_subjects))

print("TRAIN subject ID sąrašas:", train_subjects)
print("TEST subject ID sąrašas :", test_subjects)
print("TRAIN unikalių žmonių skaičius:", len(train_subjects))
print("TEST unikalių žmonių skaičius :", len(test_subjects))
print("TRAIN ir TEST subject ID sankirta:", subject_overlap)
print("Sankirta tuščia:", subject_overlap == set())

assert subject_overlap == set(), (
    f"Data leakage: tie patys subject ID yra TRAIN ir TEST: {sorted(subject_overlap)}"
)
print("Assertion praėjo: nė vienas subject nėra ir TRAIN, ir TEST.")


def class_distribution(y_part, title):
    counts = (
        y_part.merge(activity_labels, on="activity_id", how="left")
        .value_counts(["activity_id", "activity_name"])
        .rename("count")
        .reset_index()
        .sort_values("activity_id")
    )
    counts["share"] = (counts["count"] / counts["count"].sum()).round(4)
    print(f"\n{title}")
    print(counts.to_string(index=False))
    return counts


train_dist = class_distribution(y_train_final, "TRAIN klasių pasiskirstymas")
test_dist = class_distribution(y_test_final, "TEST klasių pasiskirstymas")

train_classes = set(y_train_final["activity_id"].unique())
test_classes = set(y_test_final["activity_id"].unique())
all_classes = set(activity_labels["activity_id"].tolist())

print("\nVisos 6 klasės TRAIN:", train_classes == all_classes)
print("Visos 6 klasės TEST :", test_classes == all_classes)

TRAIN subject ID sąrašas: [1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 14, 15, 17, 19, 20, 21, 22, 23, 26, 27, 30]
TEST subject ID sąrašas : [9, 10, 13, 16, 18, 24, 25, 28, 29]
TRAIN unikalių žmonių skaičius: 21
TEST unikalių žmonių skaičius : 9
TRAIN ir TEST subject ID sankirta: set()
Sankirta tuščia: True
Assertion praėjo: nė vienas subject nėra ir TRAIN, ir TEST.

TRAIN klasių pasiskirstymas
 activity_id      activity_name  count  share
           1            WALKING   1214 0.1699
           2   WALKING_UPSTAIRS   1060 0.1484
           3 WALKING_DOWNSTAIRS    970 0.1358
           4            SITTING   1233 0.1726
           5           STANDING   1322 0.1851
           6             LAYING   1345 0.1883

TEST klasių pasiskirstymas
 activity_id      activity_name  count  share
           1            WALKING    508 0.1610
           2   WALKING_UPSTAIRS    484 0.1534
           3 WALKING_DOWNSTAIRS    436 0.1382
           4            SITTING    544 0.1724
           5           STANDING   

## Skaidymo išsaugojimas

Išsaugome galutinius masyvus į `results/`, kad kitame etape nereikėtų iš naujo skaityti žalių UCI failų. Standartizavimo ir modelių čia **nedarome**.

In [7]:
results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)
out_path = results_dir / "subject_disjoint_split.npz"

np.savez_compressed(
    out_path,
    X_train_final=X_train_final.to_numpy(dtype=np.float64),
    X_test_final=X_test_final.to_numpy(dtype=np.float64),
    y_train_final=y_train_final["activity_id"].to_numpy(dtype=np.int64),
    y_test_final=y_test_final["activity_id"].to_numpy(dtype=np.int64),
    subjects_train_final=subjects_train_final["subject_id"].to_numpy(dtype=np.int64),
    subjects_test_final=subjects_test_final["subject_id"].to_numpy(dtype=np.int64),
    feature_names=np.array(feature_names),
    activity_ids=activity_labels["activity_id"].to_numpy(dtype=np.int64),
    activity_names=activity_labels["activity_name"].to_numpy(),
    random_state=np.array([RANDOM_STATE]),
)

print(f"Išsaugota: {out_path}")
print("Šio etapo grandinė baigta: subject-disjoint TRAIN/TEST paruoštas.")

Išsaugota: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\subject_disjoint_split.npz
Šio etapo grandinė baigta: subject-disjoint TRAIN/TEST paruoštas.
